### RAG and Agent Evaluation

In [2]:
import pandas as pd
# Load the ground truth data in the notebook and convert it to a list of dictionaries
df_ground_truth = pd.read_csv("ground_truth-new.csv")

In [3]:
df_ground_truth

,question,document
0,Is it okay to join the course late if I just f...,74eb249bbf
1,Can I still take this course even if I missed ...,74eb249bbf
2,If I join after the course has already started...,74eb249bbf
3,Do I need to submit my project before submissi...,74eb249bbf
4,I’m a bit late to the course—what do I need to...,74eb249bbf
...,...,...
390,Why do I get a 401 Client Error when using the...,4b30b918bc
391,What's the easiest way to force-install reques...,4b30b918bc
392,Can I install requests straight from the GitHu...,4b30b918bc
393,"If pip keeps pulling requests v2.28, what exac...",4b30b918bc


In [6]:
# Convert it to a list of dictionaries
ground_truth = df_ground_truth.to_dict(orient="records")

In [4]:
!pip install minsearch

In [7]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [42]:
# Load the FAQ documents and the search index:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [43]:
len(documents)

115

In [10]:
# Create a lookup table for the original FAQ documents:
doc_idx = {}

for doc in documents:
    doc_idx[doc["doc_id"]] = doc

In [11]:
q = ground_truth[10]
q

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [12]:
doc_idx[q["document"]]

{'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'doc_id': '489dd1c9d9'}

### Running RAG

In [ ]:
import os
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("GROQ_API_KEY")

In [ ]:
from openai import OpenAI

# Resolve API key from environment if available (OPENAI_API_KEY preferred, fallback to GROQ_API_KEY)
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY or GROQ_API_KEY environment variable before creating OpenAI client.")
openai_client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1") or os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
)

In [33]:
# For this lesson, use RAGWithUsage from the evaluation utilities. 
# It subclasses RAGBase from module 01, so it has the same rag method.

# It stores token usage after each LLM call. Then we can calculate the total cost later.

# It also uses the search boosts we selected in the search tuning 
# lesson: question=1.0, answer=2.0, and section=0.1.

from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    model="openai/gpt-oss-120b",
    index=index,
    llm_client=openai_client,
)

In [24]:
# The original question
rec = ground_truth[10]
rec

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [ ]:
question = rec["question"]

answer_llm = assistant.rag(question)
print(answer_llm)

You don’t need a Zoom link.  
The Office Hours and live workshop are streamed on **YouTube Live**.  
Before each session, the video URL is posted in the **announcements channel on Telegram and Slack** (or you can just jump to the DataTalksClub YouTube channel).  

During the stream you can ask questions via **Slido** (the link is pinned in the chat). That’s how you join and participate.


In [28]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [30]:
# Check the cost of this call:
assistant.total_cost()

0.00440925

In [25]:
rec

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'document': '489dd1c9d9'}

In [31]:
# Get the original answer from the document ID
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [32]:
# Now save both answers in one record:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': 'You don’t need a Zoom link at all.  \nThe Office Hours and live workshops are streamed on **YouTube Live**.  \n\n* The stream URL is posted in the *announcements* channel on both **Telegram** and **Slack** a few minutes before the session starts.  \n* You can also watch the live stream directly on the DataTalksClub **YouTube Channel**.  \n\nDuring the live session, ask questions through **Slido** (the link is pinned in the chat).  \n\nIf you still can’t find the link, drop a quick question in Slack and the instructor/TA will point you to it.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch li

#### Processing all questions

In [26]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [27]:
# Test it on one record:
answer_record = generate_rag_answer(ground_truth[10])
answer_record

{'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
 'answer_llm': 'You won’t be able to join via a Zoom link – that link is only shared with instructors, presenters, and TAs.  \nInstead, the session is streamed live on **YouTube** (the DataTalksClub channel) and you can:\n\n1. **Watch the live stream** on the DataTalksClub YouTube channel (https://www.youtube.com/c/DataTalksClub) or via the YouTube URL that the course team posts to the *announcements* channel on Telegram and Slack before the session starts.\n2. **Submit questions** through Slido. The Slido link is pinned in the chat during the live stream, so you can use it to ask questions in real time.\n\nSo, if you don’t have a Zoom link, just head over to the YouTube live stream and use Slido for your questions.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the cha

In [28]:
# Before running the full batch, reset the usage we collected while testing:
assistant.reset_usage()

In [29]:
# Import the parallel processing helper from the same utility file:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [36]:
# Run RAG for all ground truth questions:
with ThreadPoolExecutor(max_workers=6) as pool:
    # generate_rag_answer returns one answer record for each question.
    results = map_progress(pool, ground_truth[:10], generate_rag_answer)

  0%|          | 0/10 [00:00<?, ?it/s]

In [37]:
# Collect the answer records:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [38]:
answers

[{'question': 'Is it okay to join the course late if I just found it now?',
  'answer_llm': 'Yes—you can still join the course even if you’ve just discovered it. The only catch is that, to receive a certificate, you’ll need to submit your capstone project before the submission deadline closes.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'Can I still take this course even if I missed the start date?',
  'answer_llm': 'Yes—you can still take the course even if you missed the official start date. The videos, notebooks, and GitHub materials are always available, and you can begin whenever you like. Registration is only for gauging interest; you don’t need a confirmation email to start learning or to submit homework while the submission form remains open. If you later want a certificate, just be sure to submit your capstone project before the project‑s

In [39]:
# Calculate the total cost:
assistant.total_cost()

0.033342750000000004

In [53]:
# Save the answers:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("rag-answers-new.csv", index=False)